# Módulo 2 · Optimización Multiobjetivo con NSGA-II

Proyecto: Análisis y Diseño de Algoritmos · Optimización de Portafolios

Se implementa el algoritmo genético **NSGA-II** (Non-dominated Sorting Genetic Algorithm II)
con la librería `deap` para obtener el frente de Pareto de portafolios que maximizan el
retorno y minimizan el riesgo simultáneamente.

In [1]:
!pip install yfinance deap plotly -q

import random
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import yfinance as yf
from deap import base, creator, tools
import json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.9/866.9 kB 6.7 MB/s eta 0:00:00


## Parámetros globales y descarga de datos

`MU` y `NGEN` son el tamaño de población y el número de generaciones exigidos por la rúbrica.

In [2]:
TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
START_DATE = '2015-01-01'
END_DATE = '2024-12-31'
CAPITAL = 100000
FREQUENCY = 'Mensual'
MU = 100      # tamano de poblacion
NGEN = 80     # numero de generaciones
CXPB = 0.7    # probabilidad de cruce
MUTPB = 0.3   # probabilidad de mutacion
SEED = 42

def cargar_datos(tickers, start, end):
    raw = yf.download(tickers, start=start, end=end, auto_adjust=True, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        precios = raw['Close']
    else:
        precios = raw[['Close']]
        precios.columns = tickers
    precios = precios.dropna(how='all').ffill().dropna()
    tick_list = list(precios.columns)
    log_returns = np.log(precios / precios.shift(1)).dropna()
    mu = log_returns.mean() * 252
    cov = log_returns.cov() * 252
    return precios, log_returns, mu, cov, tick_list

precios, log_returns, mu, cov, TICK_LIST = cargar_datos(TICKERS, START_DATE, END_DATE)
retornos_simples = precios.pct_change().dropna()
print(f"Activos: {TICK_LIST}  |  Sesiones: {len(precios)}")

Activos: ['ABX.TO', 'BHP', 'BVN', 'FSM', 'VOLCABC1.LM']  |  Sesiones: 2560


## Configuración de DEAP (creator, individuos y función de evaluación)

Cada individuo es un vector de `N_ACTIVOS` genes en [0,1] que se normaliza para representar
pesos válidos (suma 1, sin ventas en corto). La fitness es multiobjetivo:
maximizar retorno, minimizar volatilidad.

In [3]:
N_ACTIVOS = len(mu)
MU_ARR, COV_ARR = mu.values, cov.values
random.seed(SEED)

if not hasattr(creator, 'FitnessMulti'):
    creator.create('FitnessMulti', base.Fitness, weights=(1.0, -1.0))
if not hasattr(creator, 'Individual'):
    creator.create('Individual', list, fitness=creator.FitnessMulti)

def _normalizar_individuo(ind, n):
    arr = np.clip(np.array(ind, dtype=float), 0.0, None)
    s = arr.sum()
    return (arr / s) if s > 0 else np.repeat(1.0 / n, n)

def evaluar(ind):
    w = _normalizar_individuo(ind, N_ACTIVOS)
    ret = float(np.dot(w, MU_ARR))
    vol = float(np.sqrt(w @ COV_ARR @ w))
    return ret, vol

toolbox = base.Toolbox()
toolbox.register('attr_float', random.random)
toolbox.register('individual', tools.initRepeat, creator.Individual, toolbox.attr_float, n=N_ACTIVOS)
toolbox.register('population', tools.initRepeat, list, toolbox.individual)
toolbox.register('evaluate', evaluar)
toolbox.register('mate', tools.cxSimulatedBinaryBounded, low=0.0, up=1.0, eta=20.0)
toolbox.register('mutate', tools.mutPolynomialBounded, low=0.0, up=1.0, eta=20.0, indpb=1.0 / N_ACTIVOS)
toolbox.register('select', tools.selNSGA2)

## Bucle evolutivo NSGA-II

Se implementa el ciclo completo: selección por torneo con distancia de crowding
(`selTournamentDCD`), cruce `cxSimulatedBinaryBounded`, mutación `mutPolynomialBounded`
y selección de la nueva generación con `selNSGA2`. Se calcula también el indicador de
**hypervolumen** (2D) en cada generación para evaluar la convergencia.

In [4]:
def hipervolumen_2d(frente, ref_ret, ref_vol):
    pts = sorted(((ind.fitness.values[0], ind.fitness.values[1]) for ind in frente), key=lambda p: p[1])
    hv, prev_vol = 0.0, ref_vol
    for ret, vol in pts:
        if vol < prev_vol:
            hv += max(0.0, prev_vol - vol) * max(0.0, ret - ref_ret)
            prev_vol = vol
    return hv

def ejecutar_nsga2(pop_size, ngen, cxpb, mutpb):
    pop_size = max(8, int(round(pop_size / 4.0)) * 4)  # multiplo de 4 (requisito de selTournamentDCD)
    pop = toolbox.population(n=pop_size)
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)
    pop = toolbox.select(pop, len(pop))

    ref_ret = min(MU_ARR.min(), 0.0) - 0.05
    ref_vol = float(np.sqrt(np.diag(COV_ARR)).max()) * 1.2

    hv_historia = []
    for _gen in range(ngen):
        offspring = tools.selTournamentDCD(pop, len(pop))
        offspring = [toolbox.clone(ind) for ind in offspring]
        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if random.random() <= cxpb:
                toolbox.mate(c1, c2)
            if random.random() <= mutpb:
                toolbox.mutate(c1)
                toolbox.mutate(c2)
            del c1.fitness.values, c2.fitness.values

        invalidos = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalidos:
            ind.fitness.values = toolbox.evaluate(ind)

        pop = toolbox.select(pop + offspring, pop_size)
        frente0 = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
        hv_historia.append(hipervolumen_2d(frente0, ref_ret, ref_vol))

    frente_final = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
    resultados = []
    for ind in frente_final:
        w = _normalizar_individuo(ind, N_ACTIVOS)
        ret, vol = ind.fitness.values
        sharpe = ret / vol if vol > 0 else 0.0
        resultados.append({'weights': w, 'ret': ret, 'vol': vol, 'sharpe': sharpe})
    resultados.sort(key=lambda r: r['vol'])
    return resultados, hv_historia

## Ejecución del algoritmo

In [5]:
frente, hv_historia = ejecutar_nsga2(MU, NGEN, CXPB, MUTPB)

rets = [r['ret'] for r in frente]
vols = [r['vol'] for r in frente]
sharpes = [r['sharpe'] for r in frente]

idx_conservador = int(np.argmin(vols))
idx_agresivo = int(np.argmax(rets))
idx_balanceado = int(np.argmax(sharpes))

print(f"Tamano del frente de Pareto: {len(frente)}")
print(f"Conservador -> retorno {rets[idx_conservador]*100:.2f}%, vol {vols[idx_conservador]*100:.2f}%")
print(f"Balanceado  -> Sharpe {sharpes[idx_balanceado]:.2f}")
print(f"Agresivo    -> retorno {rets[idx_agresivo]*100:.2f}%, vol {vols[idx_agresivo]*100:.2f}%")

Tamano del frente de Pareto: 100
Conservador -> retorno 5.03%, vol 26.72%
Balanceado  -> Sharpe 0.28
Agresivo    -> retorno 8.07%, vol 33.51%


## Visualización del frente de Pareto

Se superpone, de forma opcional, la frontera eficiente de Markowitz calculada en el
Módulo 1 (si `resultados_m1.json` ya existe en el entorno).

In [6]:
fig_pareto = px.scatter(x=vols, y=rets, color=sharpes,
                         labels={'x': 'Volatilidad', 'y': 'Retorno', 'color': 'Sharpe'},
                         color_continuous_scale='YlGnBu', title='Frente de Pareto NSGA-II')

try:
    with open('resultados_m1.json') as f:
        m1 = json.load(f)
    fig_pareto.add_scatter(x=m1['frontera_vols'], y=m1['frontera_objetivos'], mode='lines',
                            name='Frontera Markowitz', line=dict(color='#B3452F', dash='dash'))
except FileNotFoundError:
    print("Nota: ejecuta primero el notebook 1 (Datos_Markowitz) para superponer su frontera.")

fig_pareto.show()

Nota: ejecuta primero el notebook 1 (Datos_Markowitz) para superponer su frontera.


## Convergencia y portafolios representativos del frente

In [7]:
fig_hv = px.line(x=list(range(1, len(hv_historia) + 1)), y=hv_historia,
                  labels={'x': 'Generacion', 'y': 'Hypervolume Indicator'},
                  title='Evolucion del Hypervolumen')
fig_hv.show()

fig_p1 = px.pie(names=TICK_LIST, values=frente[idx_conservador]['weights'], hole=0.3,
                 title=f'Conservador (sigma={vols[idx_conservador]*100:.1f}%)')
fig_p1.show()

fig_p2 = px.pie(names=TICK_LIST, values=frente[idx_balanceado]['weights'], hole=0.3,
                 title=f'Balanceado (Sharpe={sharpes[idx_balanceado]:.2f})')
fig_p2.show()

fig_p3 = px.pie(names=TICK_LIST, values=frente[idx_agresivo]['weights'], hole=0.3,
                 title=f'Agresivo (Retorno={rets[idx_agresivo]*100:.1f}%)')
fig_p3.show()

## Simulación de riqueza de los 3 portafolios representativos y persistencia

Se simula la evolución de riqueza (rebalanceo periódico) de los portafolios Conservador,
Balanceado y Agresivo, y se guardan los resultados en `resultados_m2.json`.

In [8]:
def _fechas_rebalanceo(precios_index, freq_label):
    freq_map = {'Semanal': 'W', 'Mensual': 'M', 'Trimestral': 'Q'}
    freq = freq_map.get(freq_label, 'M')
    serie = pd.Series(precios_index, index=precios_index)
    ultimas = serie.resample(freq).last().dropna()
    return set(ultimas.values)

def simular_riqueza(retornos_simples, pesos_objetivo, capital, freq_label=None):
    pesos_objetivo = np.array(pesos_objetivo, dtype=float)
    fechas = retornos_simples.index
    rebal_dates = _fechas_rebalanceo(fechas, freq_label) if freq_label else set()
    valores_activos = capital * pesos_objetivo
    wealth = []
    for fecha, fila in retornos_simples.iterrows():
        valores_activos = valores_activos * (1 + fila.values)
        total = valores_activos.sum()
        wealth.append(total)
        if freq_label and fecha in rebal_dates:
            valores_activos = total * pesos_objetivo
    return pd.Series(wealth, index=fechas)

wealth_conservador = simular_riqueza(retornos_simples, frente[idx_conservador]['weights'], CAPITAL, freq_label=FREQUENCY)
wealth_balanceado = simular_riqueza(retornos_simples, frente[idx_balanceado]['weights'], CAPITAL, freq_label=FREQUENCY)
wealth_agresivo = simular_riqueza(retornos_simples, frente[idx_agresivo]['weights'], CAPITAL, freq_label=FREQUENCY)

def _serie_a_dict(serie):
    return {'fechas': [d.strftime('%Y-%m-%d') for d in serie.index], 'valores': serie.values.tolist()}

resultados_m2 = {
    'tickers': TICK_LIST,
    'frente_pareto': [{'weights': r['weights'].tolist(), 'ret': r['ret'], 'vol': r['vol'], 'sharpe': r['sharpe']} for r in frente],
    'hv_historia': hv_historia,
    'idx_conservador': idx_conservador, 'idx_balanceado': idx_balanceado, 'idx_agresivo': idx_agresivo,
    'wealth_conservador': _serie_a_dict(wealth_conservador),
    'wealth_balanceado': _serie_a_dict(wealth_balanceado),
    'wealth_agresivo': _serie_a_dict(wealth_agresivo),
    'mu_pop': MU, 'ngen': NGEN,
}

with open('resultados_m2.json', 'w') as f:
    json.dump(resultados_m2, f, indent=2)

print("Resultados del Modulo 2 guardados en resultados_m2.json")

/tmp/ipykernel_774/1863227330.py:5: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



Resultados del Modulo 2 guardados en resultados_m2.json
